# Q# Resource Estimator Python Interface
The resource estimator can now be imported into Python as a module using the functionality provided by [PyO3](https://pyo3.rs/v0.27.1/getting-started.html). To set it up, first install maturin in your virtual environment:

`pip install maturin`

Then build and install the module locally by running:

`run maturin develop`

After this, you should be good to go and you can import the resource estimator directly from Python.

# Functionality
The python interface offers two main functionalities:
- The first option is to input a tuple (#Qubits, #CX, #CCX) for which you want to estimate the physical resources. These values can be obtained for example from Qualtran so this provides a seemless interface between the two tools.
- The second option is to provide a Q# file which specificies a concrete algorithm for which you want to estimate the physical resources. The tool will then calculate (#Qubits, #CX, #CCX) from this Q# sharp file.

In the following I demonstrate these functionalities

In [ ]:
# Import the Rust functionality as a Python module
import qsharp_alice_bob_resource_estimator as qre

## Estimate resources from (#Qubits, #CX, #CCX)
The following example is based on the calculations in arXiv:2302.06639

In [3]:
print(qre.estimate_resources_struct.__doc__)

Estimate resources from explicit logical counts and return typed results,
optionally including a frontier of trade-offs.

# Arguments
- `qubits` — Logical (algorithm) qubit count.
- `cx` — Logical CX-equivalent two-qubit gate count.
- `ccx` — Logical CCX (Toffoli) gate count.
- `frontier` — If `true`, compute and return the frontier as structured objects.
- `error_total` — Overall error target; mutually exclusive with `error_budget`.
- `error_budget` — Tuple `(target, meas, routing)` for an explicit split.

# Returns
A tuple:
1. `EstimatesPy` — single best estimate,
2. `Vec<EstimatesPy>` — frontier (empty if `frontier == false`).

# Errors
Propagates errors from the physical resource estimator.


In [6]:
import math

bit_size = 256
window_size = 18

qubit_count = 9 * bit_size + window_size + 4

# Asymptotic gate counts, arXiv:2302.06639 (p. 21, app C.10)
cx_count = math.ceil(448 * bit_size**3 / window_size)
ccx_count = math.ceil(348 * bit_size**3 / window_size)


single, _ = qre.estimate_resources_struct(qubit_count,cx_count,ccx_count, frontier=False,error_total=None,error_budget=(0.333 * 0.5, 0.333 * 0.5, 0.0))

The elements of the result can be very easily accessed as class attributes as follows:

In [8]:
def print_summary(summary) -> None:
    """Pretty-print a resource estimate summary (no side effects beyond stdout)."""
    lines = [
        "Parameters obtained from the Rust resource estimator",
        "─────────────────────────────",
        f"# physical qubits:    {summary.physical_qubits:,}",
        f"runtime:             {summary.runtime_hours:.2f} hrs",
        f"total error:         {summary.total_error:.5f}",
        "─────────────────────────────",
        f"code distance:       {summary.code_distance} (|α|² = {summary.code_alpha2:.2f})",
        f"#factories:          {summary.factories}",
        f"factories distance:  {summary.factories_distance} (|α|² = {summary.factories_alpha2:.2f})",
        f"factory fraction:    {summary.factory_fraction_percent:.2f}%",
        "─────────────────────────────",
    ]
    print("\n".join(lines))
    
    
print_summary(single)

Parameters obtained from the Rust resource estimator
─────────────────────────────
# physical qubits:    115,203
runtime:             7.57 hrs
total error:         0.23104
─────────────────────────────
code distance:       13 (|α|² = 19.00)
#factories:          61
factories distance:  9 (|α|² = 12.83)
factory fraction:    4.50%
─────────────────────────────


## Estimate Resources from Q# File

In [9]:
print(qre.estimate_file_struct.__doc__)

Estimate resources from a Q# file and return both the best estimate and, optionally,
a frontier of Pareto-optimal trade-offs, together with the parsed logical counts.

# Arguments
- `filename` — Path to a Q# source file to be parsed and interpreted for counts.
- `frontier` — If `true`, also compute a frontier of estimates (e.g., different distances/α).
- `error_total` — Overall error target `p_total`; mutually exclusive with `error_budget`.
- `error_budget` — Tuple `(target, meas, routing)` if an explicit split is desired.

# Returns
A 3-tuple:
1. `EstimatesPy` — the single best estimate,
2. `Vec<EstimatesPy>` — optionally, the frontier (empty if `frontier == false`),
3. `LogicalCountsPy` — Python snapshot of the logical counts extracted from `filename`.

# Errors
- I/O or parsing failures when loading the Q# file,
- Failures during resource estimation.

# Notes
Counts are placed behind `Rc` to share between the estimator and the Python view
without copying the full structure.


In [12]:
filename = "qsharp/Adder.qs"
single_qsharp, frontier, counts = qre.estimate_file_struct(
    filename, 
    frontier=True, 
    error_total=None, 
    error_budget=(0.0005, 0.0005, 0.0)
)
print("\n=== Q# example ===")
print_summary(single_qsharp)


=== Q# example ===
Parameters obtained from the Rust resource estimator
─────────────────────────────
# physical qubits:    13,419
runtime:             0.00 hrs
total error:         0.00019
─────────────────────────────
code distance:       9 (|α|² = 14.00)
#factories:          2
factories distance:  5 (|α|² = 8.18)
factory fraction:    0.67%
─────────────────────────────
